# Doanh nghiệp X — Notebook đầy đủ: 3 module + 4 demo bổ sung
**Môn:** Nghiên cứu Dữ liệu lớn & Ứng dụng trong Kinh doanh — TS. Nguyễn Thôn Dã

**Cách dùng:** chạy lần lượt từng ô từ trên xuống (bấm ▶ ở mỗi ô, hoặc menu *Runtime → Run all*).
Ô cuối cùng tự nén mọi kết quả thành `ket_qua_tieu_luan.zip` và tải về máy.

| Ô | Nội dung | Dùng cho phần nào của bài |
|---|---|---|
| 0–2 | Cài đặt, tải dữ liệu, làm sạch | Bảng 3.1 |
| 3 | Module 1: so sánh 8 mô hình | Bảng 4.2, Hình 4.3 |
| 4 | Module 1: kế hoạch cắt + **Demo 3** | Bảng 4.4, Hình 4.4 |
| 5 | Module 2: FP-Growth | Bảng 4.6, Hình 4.5 |
| 6 | Module 3: giá trị đơn | Bảng 4.7, 4.8 |
| 7 | Kiểm tra lại các con số trong bài | Mục 4.1, Bảng 4.3 |
| 8 | **Demo 1**: kiểm định Diebold–Mariano | Bảng 4.3b (mới) |
| 9 | **Demo 4**: thực tế vs dự báo | Hình 4.3b (mới) |
| 10 | **Demo 2**: khả năng mở rộng Spark | Phụ lục D (mới) |
| 11 | In sẵn đoạn văn để dán vào bài | Chương 4, Phụ lục D |
| 12 | Nén và tải kết quả | — |

## Ô 0 — Cài đặt thư viện (chạy 1 lần, khoảng 2–3 phút)

In [ ]:
!pip install -q pyspark==3.5.3 statsforecast mlforecast lightgbm openpyxl

## Ô 1 — Tải file đơn hàng lên và khởi động Spark
Bấm **Choose Files**, chọn file Excel đơn hàng (file gốc chị vẫn dùng cho notebook cũ).

In [ ]:
import os, re, unicodedata, itertools, warnings, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
try:
    from google.colab import files
    up = files.upload(); XLSX = list(up.keys())[0]; IN_COLAB = True
except ImportError:                       # chạy ngoài Colab
    XLSX = os.environ.get("XLSX", "orders.xlsx"); IN_COLAB = False
OUT = "ket_qua"; os.makedirs(OUT, exist_ok=True)
TOP_N, TEST_WEEKS = 30, 12
NAVY, ORANGE = "#1F3A5F", "#E07B39"

# Khởi động Spark MỘT lần, cấp 6 GB bộ nhớ để Demo 2 không bị thiếu RAM
from pyspark.sql import SparkSession, functions as F, Window as W
spark = (SparkSession.builder.appName("DoanhNghiepX").master("local[*]")
         .config("spark.driver.memory", "6g").config("spark.sql.shuffle.partitions", "16").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("File:", XLSX, "| Spark", spark.version)

## Ô 2 — Làm sạch dữ liệu, tách style / màu / size

In [ ]:
def cs(x):
    if x is None: return None
    x = x.strip().upper().replace("SIZE", "").strip().lstrip(":").strip().replace("FREESIZE", "FREE")
    x = re.split(r"\s", x)[0] if x else x
    return x or None
def parse(s):
    if s is None: return ("?", None, None)
    s = unicodedata.normalize("NFC", str(s)).strip()
    if s == "": return ("?", None, None)
    if s.upper().replace(" ", "") == "COD": return ("COD", None, None)
    m = re.search(r"^(.*?)\s*/\s*màu\s*:\s*(.*?)(?:\s*/\s*size\s*:\s*(.*?))?\s*$", s, flags=re.IGNORECASE)
    if m: return (m.group(1).strip().upper(), (m.group(2) or "").strip().upper() or None, cs(m.group(3)))
    if s.count("/") >= 1:
        p = [x.strip() for x in s.split("/")]
        size = cs(p[-1]) if len(p) >= 2 else None
        color = p[-2].strip().upper() if len(p) >= 3 else None
        nm = re.sub(r"^\d{6,}\s*-\s*", "", p[0]); mc = re.search(r"\(([^)]+)\)", nm)
        code_ = mc.group(1).strip().upper() if mc else None
        nm = re.sub(r"^[^-]*-\s*\([^)]*\)\s*-?\s*", "", nm).strip(); nm = re.sub(r"\(([^)]+)\)", "", nm).strip()
        return (code_ or (nm.upper() if nm else s.upper()), color, size)
    return (s.upper(), None, None)

df = pd.read_excel(XLSX)[["STT", "Giá trị đơn hàng sau giảm giá", "Ngày tạo đơn", "Sản phẩm"]]
df.columns = ["stt", "aov", "ngay", "sp"]
df["is_head"] = pd.to_numeric(df["stt"], errors="coerce").notna()
df["oid"] = df["is_head"].cumsum()
df["ngay"] = df["ngay"].where(df["ngay"].astype(str).str.strip().ne("")).ffill()
df["aov"] = pd.to_numeric(df["aov"], errors="coerce").where(df["is_head"]).ffill()
df["dt"] = pd.to_datetime(df["ngay"], format="%d/%m/%Y", errors="coerce")
d = df.dropna(subset=["sp"]).copy(); d = d[d["sp"].astype(str).str.strip().ne("")]
pr = d["sp"].map(parse)
d["style"] = pr.map(lambda t: t[0]); d["color"] = pr.map(lambda t: t[1]); d["size"] = pr.map(lambda t: t[2])
d = d.dropna(subset=["dt"])
d["ds"] = (d["dt"] - pd.to_timedelta(d["dt"].dt.weekday, unit="D")).dt.normalize()
print("ETL OK: %d dòng | %d đơn | %d tuần | %d style" % (len(d), d["oid"].nunique(), d["ds"].nunique(), d["style"].nunique()))

## Ô 3 — Module 1: so sánh 8 mô hình dự báo (kiểm định lùi 12 tuần)
Ô này chạy lâu nhất (khoảng 3–6 phút).

In [ ]:
top = d["style"].value_counts().head(TOP_N).index.tolist()
weekly = d[d["style"].isin(top)].groupby(["style", "ds"]).size().rename("y").reset_index()
weeks = pd.date_range(d["ds"].min(), d["ds"].max(), freq="W-MON")
panel = (pd.DataFrame(list(itertools.product(top, weeks)), columns=["unique_id", "ds"])
         .merge(weekly.rename(columns={"style": "unique_id"}), on=["unique_id", "ds"], how="left")
         .fillna({"y": 0}).sort_values(["unique_id", "ds"]))
panel["y"] = panel["y"].astype(float)

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, WindowAverage, CrostonClassic, CrostonSBA, TSB, AutoETS
sf = StatsForecast(models=[Naive(), WindowAverage(window_size=4), SeasonalNaive(season_length=52),
                           CrostonClassic(), CrostonSBA(), TSB(alpha_d=0.2, alpha_p=0.2), AutoETS(season_length=52)],
                   freq="W-MON", n_jobs=1)
cv = sf.cross_validation(df=panel, h=1, step_size=1, n_windows=TEST_WEEKS).reset_index()

from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
import lightgbm as lgb
lgbm = lgb.LGBMRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, num_leaves=15,
                         min_child_samples=20, subsample=0.8, verbosity=-1)
mlf = MLForecast(models={"LightGBM": lgbm}, freq="W-MON", lags=[1, 2, 4, 8],
                 lag_transforms={1: [RollingMean(window_size=4), RollingMean(window_size=8)]},
                 date_features=["week", "month"])
cvm = mlf.cross_validation(df=panel, h=1, step_size=1, n_windows=TEST_WEEKS).reset_index()

def scr(cvv, c):
    e = cvv["y"] - cvv[c]; dd = cvv["y"].abs() + cvv[c].abs()
    return np.sqrt((e**2).mean()), e.abs().mean(), float(np.where(dd == 0, 0, 2*e.abs()/dd.where(dd != 0, 1)).mean()*100)
nm = {"Naive": "Naive (tuần trước)", "WindowAverage": "Trung bình trượt 4T", "SeasonalNaive": "Seasonal Naive",
      "CrostonClassic": "Croston", "CrostonSBA": "Croston-SBA", "TSB": "TSB", "AutoETS": "AutoETS"}
rows = [(nm.get(c, c),) + scr(cv, c) for c in cv.columns if c not in ["unique_id", "ds", "cutoff", "y", "index"]]
rows.append(("LightGBM (M5)",) + scr(cvm, "LightGBM"))
scores = pd.DataFrame(rows, columns=["Mô hình", "RMSE", "MAE", "sMAPE"]).sort_values("RMSE").round(2)
scores.to_excel(f"{OUT}/M1_ket_qua_sai_so.xlsx", index=False)
print(scores.to_string(index=False))

## Ô 4 — Module 1: kế hoạch cắt tuần tới + **Demo 3** (phân bổ không hụt do làm tròn)

In [ ]:
mlf.fit(panel)
fc = mlf.predict(h=1).rename(columns={"unique_id": "style", "LightGBM": "fc_style"})
fc["fc_style"] = fc["fc_style"].clip(lower=0).round()
NEXT_WEEK = (weeks[-1] + pd.Timedelta(days=7)).strftime("%d/%m/%Y")
last8 = weeks[-8]
sh = d[(d["ds"] >= last8) & (d["style"].isin(top))].groupby(["style", "color", "size"]).size().rename("n").reset_index()
sh["tot"] = sh.groupby("style")["n"].transform("sum"); sh["share"] = sh["n"] / sh["tot"]

# Cách cũ: làm tròn từng ô (bị hụt)
plan = sh.merge(fc[["style", "fc_style"]], on="style")
plan["cut_qty"] = (plan["share"] * plan["fc_style"]).round()
plan = plan[plan["cut_qty"] > 0][["style", "color", "size", "cut_qty"]].sort_values("cut_qty", ascending=False)
plan.to_excel(f"{OUT}/M1_ke_hoach_cat.xlsx", index=False)

# DEMO 3: phân bổ theo phần dư lớn nhất -> tổng cắt mỗi style = dự báo
plan2 = sh.merge(fc[["style", "fc_style"]], on="style")
def alloc(g):
    total = int(g["fc_style"].iloc[0]); raw = g["share"] * total
    q = np.floor(raw); rem = total - int(q.sum())
    idx = (raw - q).sort_values(ascending=False).index[:max(rem, 0)]
    q.loc[idx] += 1; return q
plan2["cut_qty"] = plan2.groupby("style", group_keys=False)[["share", "fc_style"]].apply(alloc).astype(int)
plan2 = plan2[plan2["cut_qty"] > 0][["style", "color", "size", "cut_qty"]].sort_values("cut_qty", ascending=False)
plan2.to_excel(f"{OUT}/Demo3_ke_hoach_cat_khop_du_bao.xlsx", index=False)

fc_total = fc["fc_style"].sum(); fc_plan_styles = fc[fc["style"].isin(plan2["style"])]["fc_style"].sum()
print(f"Tuần dự báo: {NEXT_WEEK} | Tổng dự báo 30 chuỗi: {fc_total:.0f}")
print(f"Cách cũ : {len(plan)} dòng, tổng {plan['cut_qty'].sum():.0f} sp thuộc {plan['style'].nunique()} style")
print(f"Demo 3  : {len(plan2)} dòng, tổng {plan2['cut_qty'].sum():.0f} sp thuộc {plan2['style'].nunique()} style "
      f"(= tổng dự báo của các style này: {fc_plan_styles:.0f})")
print(plan2.head(10).to_string(index=False))

## Ô 5 — Module 2: combo bằng FP-Growth (Spark MLlib)

In [ ]:
from pyspark.ml.fpm import FPGrowth
bk = d[d["style"] != "COD"].groupby("oid")["style"].apply(lambda s: sorted(set(s)))
bk = bk[bk.map(len) >= 1]
sdf = spark.createDataFrame([(int(o), list(v)) for o, v in bk.items()], ["oid", "items"])
fp = FPGrowth(itemsCol="items", minSupport=0.002, minConfidence=0.05).fit(sdf)
r = (fp.associationRules.filter((F.size("antecedent") == 1) & (F.size("consequent") == 1))
     .withColumn("A", F.col("antecedent")[0]).withColumn("B", F.col("consequent")[0])
     .withColumn("pair", F.array_sort(F.array("A", "B"))))
r = r.withColumn("rn", F.row_number().over(W.partitionBy("pair").orderBy(F.desc("lift")))).filter("rn=1")
combo = r.orderBy(F.desc("lift")).select("A", "B", "support", "confidence", "lift").limit(12).toPandas().round(4)
N_BASKET = len(bk); combo["so_don"] = (combo["support"] * N_BASKET).round().astype(int)
combo.to_excel(f"{OUT}/M2_combo.xlsx", index=False)
print("Số giỏ hàng:", N_BASKET); print(combo.to_string(index=False))

## Ô 6 — Module 3: giá trị đơn (AOV) và mô phỏng chính sách

In [ ]:
ov = d.groupby("oid")["aov"].first().dropna(); nit = d.groupby("oid").size()
ppi = (ov / nit).median(); base = ov.sum(); n1 = int((nit == 1).sum())
combo_up = n1 * 0.15 * ppi * 0.9
thr = 300000; near = ov[(ov >= thr*0.7) & (ov < thr)]; thr_up = len(near) * (thr - near.mean()) * 0.30
m3 = pd.DataFrame({"Chỉ số": ["AOV trung bình", "AOV trung vị", "% đơn 1 món", "AOV đơn 1 món",
                              "% đơn nhiều món", "AOV đơn nhiều món", "Combo uplift %", "Ngưỡng uplift %"],
                   "Giá trị": [ov.mean(), ov.median(), (nit == 1).mean()*100, ov[nit == 1].mean(),
                               (nit >= 2).mean()*100, ov[nit >= 2].mean(), combo_up/base*100, thr_up/base*100]}).round(2)
m3.to_excel(f"{OUT}/M3_aov.xlsx", index=False)
print(m3.to_string(index=False))

## Ô 7 — Kiểm tra lại các con số trong bài
Ô này tính lại các số ở Bảng 3.1, mục 4.1, Bảng 4.3 (WAPE) và so với số đang ghi trong bài.
Cột **Khớp?** = ✓ nghĩa là số trong bài đúng; ✗ thì sửa số trong bài theo cột **Tính lại**. Riêng *số mã hàng chi tiết* và *tháng cao/thấp nhất* có thể lệch nhẹ do cách đếm; chỉ sửa khi lệch nhiều.

In [ ]:
nit_style = d.groupby("oid")["style"].nunique()
cnt = d["style"].value_counts(); share = cnt.cumsum() / cnt.sum() * 100
cnt_nc = cnt.drop("COD", errors="ignore"); share_nc = cnt_nc.cumsum() / cnt_nc.sum() * 100
wk = d.groupby("ds").size(); mon = d.groupby(d["dt"].dt.to_period("M")).size()
m = cv[["unique_id", "ds", "y", "Naive"]].merge(cvm[["unique_id", "ds", "LightGBM"]], on=["unique_id", "ds"])
m["LightGBM"] = m["LightGBM"].clip(lower=0)
def wape(sub, c): return (sub["y"] - sub[c]).abs().sum() / sub["y"].sum() * 100
agg = m.groupby("ds")[["y", "Naive", "LightGBM"]].sum()
checks = [
 ("Số dòng bán", len(d), 17642), ("Số đơn hàng", d["oid"].nunique(), 8460),
 ("Số tuần có đơn", d["ds"].nunique(), 60), ("Số tuần lịch", len(weeks), 62),
 ("Số dòng sản phẩm (style)", d["style"].nunique(), 585),
 ("Số mã hàng chi tiết (style–màu–size)", len(d[["style", "color", "size"]].drop_duplicates()), 3244),
 ("Tỷ trọng COD (%)", (d["style"] == "COD").mean()*100, 10.9),
 ("Số món trung bình/đơn", nit.mean(), 2.09), ("% đơn ≥ 2 món", (nit >= 2).mean()*100, 62.8),
 ("% đơn ≥ 2 dòng sản phẩm", (nit_style >= 2).mean()*100, 45.0),
 ("Doanh số TB tuần có đơn", wk.mean(), 294), ("Tháng cao nhất (dòng)", mon.max(), 1881), ("Tháng thấp nhất (dòng)", mon.min(), 355),
 ("Top 10 style (% có COD)", share.iloc[9], 37.4), ("Top 30 style (% có COD)", share.iloc[29], 52.9),
 ("Top 10 style (% bỏ COD)", share_nc.iloc[9], 31.2), ("Top 30 style (% bỏ COD)", share_nc.iloc[29], 47.8),
 ("% ô bằng 0 (30 chuỗi)", (panel["y"] == 0).mean()*100, 58.9),
 ("Nhu cầu TB mỗi ô kiểm định", m["y"].mean(), 6.3),
 ("WAPE mọi ô – Naive", wape(m, "Naive"), 74.4), ("WAPE mọi ô – LightGBM", wape(m, "LightGBM"), 76.3),
 ("WAPE ô có bán – Naive", wape(m[m.y > 0], "Naive"), 71.5), ("WAPE ô có bán – LightGBM", wape(m[m.y > 0], "LightGBM"), 64.0),
 ("WAPE tổng 30 chuỗi – Naive", wape(agg, "Naive"), 22.1), ("WAPE tổng 30 chuỗi – LightGBM", wape(agg, "LightGBM"), 27.7),
]
chk = pd.DataFrame(checks, columns=["Chỉ tiêu", "Tính lại", "Trong bài"])
chk["Tính lại"] = chk["Tính lại"].astype(float).round(2)
chk["Khớp?"] = np.where((chk["Tính lại"] - chk["Trong bài"]).abs() <= np.maximum(0.06, chk["Trong bài"].abs()*0.005), "✓", "✗")
chk.to_excel(f"{OUT}/Kiem_tra_so_lieu.xlsx", index=False)
print(chk.to_string(index=False))

## Ô 8 — Demo 1: kiểm định Diebold–Mariano (LightGBM có thật sự tốt hơn Naive?)

In [ ]:
from scipy import stats
def dm(dd):
    n = len(dd); s = dd.mean() / np.sqrt(dd.var(ddof=1) / n)
    p = 2 * (1 - stats.t.cdf(abs(s), df=n - 1)); return round(float(s), 3), round(float(p), 4), n
rows = []
for name, loss in [("Sai số tuyệt đối", lambda e: e.abs()), ("Sai số bình phương", lambda e: e**2)]:
    dd = loss(m["y"] - m["LightGBM"]) - loss(m["y"] - m["Naive"])   # âm = LightGBM tốt hơn
    rows.append((name, "Theo ô (360)", *dm(dd)))
    rows.append((name, "Gộp theo tuần (12)", *dm(dd.groupby(m["ds"]).mean())))
dmres = pd.DataFrame(rows, columns=["Hàm tổn thất", "Cấp", "Thống kê DM", "p-value", "n"])
dmres.to_excel(f"{OUT}/Demo1_DM_test.xlsx", index=False)
print(dmres.to_string(index=False))
print("\nĐọc kết quả: DM âm = LightGBM sai ít hơn Naive; p-value < 0,05 = khác biệt có ý nghĩa thống kê.")

## Ô 9 — Demo 4: biểu đồ thực tế và dự báo trong 12 tuần kiểm định

In [ ]:
import matplotlib.dates as mdates
pick = plan2.groupby("style")["cut_qty"].sum().nlargest(2).index.tolist()   # 2 dòng lớn nhất trong kế hoạch cắt (V36, Q93)
def nice(s): return " ".join(w.upper() if any(ch.isdigit() for ch in w) else w for w in s.capitalize().split())
fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.3))
for ax, s in zip(axs, pick):
    g = m[m["unique_id"] == s].sort_values("ds")
    ax.plot(g["ds"], g["y"], "o-", color="#222", lw=1.8, label="Thực tế")
    ax.plot(g["ds"], g["LightGBM"], "s--", color=ORANGE, label="LightGBM")
    ax.plot(g["ds"], g["Naive"], "^:", color=NAVY, label="Naive")
    ax.set_title(nice(s), fontsize=10, fontweight="bold"); ax.grid(alpha=.3)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m")); ax.tick_params(axis="x", rotation=45, labelsize=8)
axs[0].set_ylabel("Số sản phẩm/tuần"); axs[0].legend(fontsize=8, frameon=False)
fig.tight_layout(); fig.savefig(f"{OUT}/Demo4_thuc_te_du_bao.png", dpi=200); plt.show()
e4 = {s: (m[m.unique_id == s]["y"] - m[m.unique_id == s]["LightGBM"]).abs().mean() for s in pick}
e4n = {s: (m[m.unique_id == s]["y"] - m[m.unique_id == s]["Naive"]).abs().mean() for s in pick}
for s in pick: print(f"{s}: MAE LightGBM = {e4[s]:.2f} | MAE Naive = {e4n[s]:.2f}")

## Ô 10 — Demo 2: thử khả năng mở rộng trên Spark
Nhân bản dữ liệu thật lên 1, 5, 20, 50 lần rồi đo thời gian Spark. Mất khoảng 3–8 phút.
Nếu báo lỗi hết bộ nhớ ở mức 50, sửa dòng `FACTORS = [1, 5, 20]` rồi chạy lại ô này.

In [ ]:
FACTORS = [1, 5, 20, 50]
base_df = d[["oid", "style", "ds"]].copy(); base_df["ds"] = base_df["ds"].dt.strftime("%Y-%m-%d")
sbase = spark.createDataFrame(base_df).cache(); sbase.count()
res = []
for k in FACTORS:
    big = (sbase.crossJoin(spark.range(k).withColumnRenamed("id", "rep"))
                .withColumn("oid2", F.concat_ws("_", F.col("oid").cast("string"), F.col("rep").cast("string")))).cache()
    n = big.count()
    t0 = time.time(); big.groupBy("style", "ds").count().count(); t1 = time.time()
    bkk = big.filter(F.col("style") != "COD").groupBy("oid2").agg(F.collect_set("style").alias("items"))
    fpk = FPGrowth(itemsCol="items", minSupport=0.002, minConfidence=0.05).fit(bkk)
    nrule = fpk.associationRules.count(); t2 = time.time()
    res.append((k, n, round(t1 - t0, 1), round(t2 - t1, 1), nrule)); big.unpersist()
    print(f"x{k}: {n:,} dòng | tổng hợp tuần {t1-t0:.1f}s | FP-Growth {t2-t1:.1f}s | {nrule} luật")
sc = pd.DataFrame(res, columns=["Hệ số nhân", "Số dòng", "Tổng hợp tuần (giây)", "FP-Growth (giây)", "Số luật"])
sc.to_excel(f"{OUT}/Demo2_scalability.xlsx", index=False)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(sc["Số dòng"]/1e6, sc["Tổng hợp tuần (giây)"], "o-", color=NAVY, label="Tổng hợp theo tuần")
ax.plot(sc["Số dòng"]/1e6, sc["FP-Growth (giây)"], "s-", color=ORANGE, label="FP-Growth")
for _, rr in sc.iterrows(): ax.annotate(f"×{int(rr['Hệ số nhân'])}", (rr["Số dòng"]/1e6, rr["FP-Growth (giây)"]), textcoords="offset points", xytext=(4, 6), fontsize=8)
ax.set_xlabel("Số dòng giao dịch (triệu)"); ax.set_ylabel("Thời gian chạy (giây)")
ax.grid(alpha=.3); ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(f"{OUT}/Demo2_scalability.png", dpi=200); plt.show()
CPU = os.cpu_count()
try:
    RAM = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1)
except Exception:
    RAM = None
print(sc.to_string(index=False)); print(f"Cấu hình máy: {CPU} nhân CPU, {RAM} GB RAM")

## Ô 11 — In sẵn các đoạn văn để dán vào bài (số đã điền tự động)

In [ ]:
def vn(x, k=2): return f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
def pv(p): return "< 0,001" if p < 0.001 else "= " + vn(p, 3)
ra, rs = dmres.iloc[0], dmres.iloc[2]     # sai số tuyệt đối / bình phương, theo ô
rw_a, rw_s = dmres.iloc[1], dmres.iloc[3] # theo tuần
sig_s = rs["p-value"] < 0.05
sig_a = ra["p-value"] < 0.05
print("=" * 90); print("[A] CHÈN SAU BẢNG 4.3 (mục 4.2) — đoạn văn + Bảng 4.3b"); print("=" * 90)
txt = (f"Để kiểm tra chênh lệch giữa LightGBM và naive có phải do ngẫu nhiên hay không, tác giả dùng kiểm định "
       f"Diebold–Mariano (Diebold & Mariano, 1995) trên 360 cặp dự báo. Với sai số bình phương, thống kê DM bằng "
       f"{vn(rs['Thống kê DM'])} (p {pv(rs['p-value'])}); với sai số tuyệt đối, DM bằng {vn(ra['Thống kê DM'])} (p {pv(ra['p-value'])}). ")
def huong(st): return "LightGBM tốt hơn" if st < 0 else "naive tốt hơn"
if sig_s and sig_a:
    txt += f"Cả hai chênh lệch đều có ý nghĩa thống kê ở mức 5% (sai số bình phương: {huong(rs['Thống kê DM'])}; sai số tuyệt đối: {huong(ra['Thống kê DM'])}). "
elif sig_s:
    txt += f"Chênh lệch về sai số bình phương có ý nghĩa thống kê ở mức 5% ({huong(rs['Thống kê DM'])}), còn chênh lệch về sai số tuyệt đối thì không. "
elif sig_a:
    txt += f"Chênh lệch về sai số tuyệt đối có ý nghĩa thống kê ở mức 5% ({huong(ra['Thống kê DM'])}), còn chênh lệch về sai số bình phương thì không. "
else:
    txt += "Cả hai chênh lệch đều không có ý nghĩa thống kê ở mức 5%, nên trên dữ liệu hiện có chưa đủ bằng chứng để khẳng định LightGBM chính xác hơn naive. "
txt += (f"Khi gộp theo tuần để tính đến tương quan giữa các chuỗi trong cùng một tuần (12 quan sát), p-value lần lượt là "
        f"{vn(rw_s['p-value'], 3)} và {vn(rw_a['p-value'], 3)} (Bảng 4.3b).")
print(txt); print("\nBảng 4.3b. Kiểm định Diebold–Mariano giữa LightGBM và naive"); print(dmres.to_string(index=False))

print("\n" + "=" * 90); print("[B] CHÈN SAU BẢNG 4.3b — đoạn văn + Hình 4.3b (file Demo4_thuc_te_du_bao.png)"); print("=" * 90)
print(f"Hình 4.3b so sánh thực tế với hai dự báo của {nice(pick[0])} và {nice(pick[1])} trong 12 tuần kiểm định. "
      f"MAE của LightGBM ở hai dòng này lần lượt là {vn(e4[pick[0]])} và {vn(e4[pick[1]])}, so với {vn(e4n[pick[0]])} và {vn(e4n[pick[1]])} của naive. "
      "Naive luôn đi trễ thực tế một tuần, còn LightGBM cho đường dự báo mượt hơn nhưng không bắt kịp các tuần tăng đột biến.")
print("Chú thích: Hình 4.3b. Thực tế và dự báo trong 12 tuần kiểm định của hai dòng sản phẩm bán chạy nhất")
print("Nguồn: Tính toán của tác giả.")
print("(Câu cuối là mô tả gợi ý — chị nhìn hình và sửa lại cho đúng với đường thực tế.)")

print("\n" + "=" * 90); print("[C] THAY ĐOẠN SAU BẢNG 4.4 (mục 4.2) — Demo 3"); print("=" * 90)
print(f"Nếu làm tròn riêng từng ô, bảng kế hoạch chỉ còn {plan['cut_qty'].sum():.0f} sản phẩm, thấp hơn tổng dự báo vì các ô dưới 0,5 sản phẩm bị làm tròn về 0. "
      f"Để tổng số lượng cắt của mỗi dòng sản phẩm khớp với dự báo, tác giả dùng phương pháp phân bổ theo phần dư lớn nhất: lấy phần nguyên của từng ô trước, "
      f"phần còn thiếu được cộng cho các ô có phần thập phân lớn nhất. Sau điều chỉnh, bảng gồm {len(plan2)} tổ hợp màu – kích cỡ thuộc {plan2['style'].nunique()} dòng sản phẩm, "
      f"tổng {plan2['cut_qty'].sum():.0f} sản phẩm, bằng đúng tổng dự báo của các dòng này. COD và các dòng không ghi kích cỡ vẫn nằm ngoài bảng.")

print("\n" + "=" * 90); print("[D] PHỤ LỤC D (mới) + một câu thêm vào cuối mục 2.2 — Demo 2"); print("=" * 90)
lo, hi = sc.iloc[0], sc.iloc[-1]
print("Phụ lục D. Thử nghiệm khả năng mở rộng của quy trình trên Spark")
print(f"Để kiểm tra khả năng mở rộng, dữ liệu giao dịch được nhân bản lần lượt {', '.join(str(int(k)) for k in sc['Hệ số nhân'])} lần, mỗi bản sao được xem như một tập đơn hàng mới. "
      f"Với mỗi mức, tác giả đo thời gian Spark tổng hợp doanh số theo tuần và chạy FP-Growth trên Google Colab ({CPU} nhân CPU, khoảng {vn(RAM, 1) if RAM else '?'} GB RAM). "
      f"Khi số dòng tăng từ {vn(lo['Số dòng'], 0)} lên {vn(hi['Số dòng'], 0)}, thời gian FP-Growth tăng từ {vn(lo['FP-Growth (giây)'], 1)} lên {vn(hi['FP-Growth (giây)'], 1)} giây, "
      f"còn số luật kết hợp thu được {'không đổi' if sc['Số luật'].nunique() == 1 else 'thay đổi từ ' + str(int(lo['Số luật'])) + ' thành ' + str(int(hi['Số luật']))} (Bảng D.1, Hình D.1). "
      "Kết quả cho thấy cùng một đoạn mã xử lý được lượng dữ liệu lớn hơn nhiều lần mà không phải viết lại.")
print("\nBảng D.1. Thời gian chạy theo quy mô dữ liệu"); print(sc.to_string(index=False))
print("Hình D.1. Thời gian chạy của Spark khi tăng quy mô dữ liệu (file Demo2_scalability.png)")
print("\nCâu thêm vào cuối mục 2.2: "
      f"Thử nghiệm ở Phụ lục D cho thấy khi nhân dữ liệu lên {int(hi['Hệ số nhân'])} lần ({vn(hi['Số dòng'], 0)} dòng), "
      f"quy trình vẫn chạy trên một máy Colab miễn phí trong khoảng {vn(hi['FP-Growth (giây)'] + hi['Tổng hợp tuần (giây)'], 0)} giây.")

print("\n" + "=" * 90); print("[E] TÀI LIỆU THAM KHẢO cần thêm"); print("=" * 90)
print("Diebold, F. X., & Mariano, R. S. (1995). Comparing predictive accuracy. Journal of Business & Economic Statistics, 13(3), 253–263.")

## Ô 12 — Nén và tải toàn bộ kết quả về máy

In [ ]:
import shutil
shutil.make_archive("ket_qua_tieu_luan", "zip", OUT)
print("Đã nén:", sorted(os.listdir(OUT)))
if IN_COLAB:
    files.download("ket_qua_tieu_luan.zip")